# 01 — Data Cleaning

**Input:** `../data/EGT Project (Responses).csv` (raw Google Form export)  
**Output:** `../data/cleaned_responses.csv`

Steps:
1. Load raw CSV; rename verbose Google-Form columns to short snake_case.
2. Normalize `What is your major` → `major` (primary only).
3. Encode ordinal text → numeric: `w` (GPA importance, 1–5), `q7_score` (0–2), `q9_score` (−2..2).
4. Compute `s = layups / courses` (layup ratio over last 3 on-campus terms).
5. Save cleaned dataset.

## Major-normalization rules

| Pattern | Rule | Example |
|---|---|---|
| `X modified with Y` / `X mod Y` / `X w Y` | Primary = X (Dartmouth modifier convention) | `Math mod Econ` → `Mathematics` |
| `X & Y` / `X + Y` / `X and Y` / `X double major` | Primary = first-listed X, row flagged `is_double=True` | `CS & Math` → `Computer Science` |
| Abbreviation/whitespace | Map to canonical | `Econ` / `ECon` / `Economics ` → `Economics` |

`is_double=True` is used in `02_empirics.ipynb` as a robustness filter.

In [1]:
import re
import pandas as pd
import numpy as np

RAW_PATH = "../data/EGT Project (Responses).csv"
OUT_PATH = "../data/cleaned_responses.csv"

In [2]:
df = pd.read_csv(RAW_PATH)
df.columns = [
    "timestamp",
    "year",
    "major_raw",
    "track",
    "courses",
    "layups",
    "strategy_self",
    "q7_avoided",
    "gpa_importance",
    "q9_env",
    "q10_counterfactual",
]
print(f"Loaded {len(df)} rows × {len(df.columns)} cols")
df.head()

Loaded 42 rows × 11 cols


,timestamp,year,major_raw,track,courses,layups,strategy_self,q7_avoided,gpa_importance,q9_env,q10_counterfactual
0,5/7/2026 13:27:53,28,Mathematics,Tech/engineering,9,1,"I choose based on genuine interest, regardless...",Never,Very important,Neutral,Would not change
1,5/10/2026 11:30:06,27,"Economics modified with Computer Science, Germ...",Finance/consulting,9,5,"I lean toward expected grade, with interest as...",Never,Very important,Strongly yes,It would not change.
2,5/10/2026 11:30:28,29,Economics,Finance/consulting,9,2,I choose almost entirely based on expected gra...,"Yes, once or twice",Very important,Strongly yes,I would take more interesting classes that ben...
3,5/10/2026 11:30:53,27,Gov and Econ double major,Finance/consulting,9,2,"I choose based on genuine interest, regardless...",Never,Moderately important,Somewhat yes,I wouldn’t
4,5/10/2026 11:32:18,28,Computer Science,Finance/consulting,9,4,"I lean toward expected grade, with interest as...","Yes, once or twice",Very important,Somewhat yes,If there were no enforced medians (or they wer...


## Major normalization

In [3]:
ABBREVIATION_MAP = {
    "math": "Mathematics",
    "mathematics": "Mathematics",
    "econ": "Economics",
    "economics": "Economics",
    "cs": "Computer Science",
    "computer science": "Computer Science",
    "gov": "Government",
    "government": "Government",
    "qss": "QSS",
    "biology": "Biology",
    "biochem": "Biochemistry",
    "biochemistry": "Biochemistry",
    "engineering": "Engineering",
    "engineering sciences": "Engineering",
    "earth sciences": "Earth Sciences",
    "envs": "Environmental Studies",
    "environmental studies": "Environmental Studies",
    "religion": "Religion",
    "ppe": "PPE",
    "history": "History",
    "physics": "Physics",
}

# Match modifier connectives: "modified with", "mod" / "modified", "w" / "w/".
MOD_RE = re.compile(r"\b(?:modified\s+with|modified|mod|w/?)\b", re.IGNORECASE)
# Match double-major connectives: &, +, comma, or " and " (with explicit spaces).
DOUBLE_SPLIT_RE = re.compile(r"\s*(?:&|\+|,)\s*|\s+and\s+", re.IGNORECASE)

def clean_major(raw):
    if pd.isna(raw):
        return pd.Series([None, False], index=["major", "is_double"])
    s = str(raw).strip()
    if not s:
        return pd.Series([None, False], index=["major", "is_double"])

    is_double = bool(re.search(r"\bdouble\s+major\b", s, flags=re.IGNORECASE))
    s_clean = re.sub(r"\bdouble\s+major\b", "", s, flags=re.IGNORECASE).strip()

    # Modifier rule takes precedence: 'Economics modified with CS' → Economics.
    mod_parts = MOD_RE.split(s_clean, maxsplit=1)
    if len(mod_parts) > 1 and mod_parts[0].strip():
        primary_raw = mod_parts[0].strip()
    else:
        # Double-major rule: take first listed.
        parts = [p for p in DOUBLE_SPLIT_RE.split(s_clean) if p and p.strip()]
        if len(parts) > 1:
            is_double = True
            primary_raw = parts[0].strip()
        else:
            primary_raw = s_clean

    key = primary_raw.lower().strip()
    canonical = ABBREVIATION_MAP.get(key, primary_raw.title())
    return pd.Series([canonical, is_double], index=["major", "is_double"])

df[["major", "is_double"]] = df["major_raw"].apply(clean_major)

In [4]:
# Spot check: show the raw → cleaned mapping for every row.
print("Row-by-row mapping (raw → cleaned, is_double):")
for i, row in df[["major_raw", "major", "is_double"]].iterrows():
    flag = " *double*" if row["is_double"] else ""
    print(f"  row {i:2d}: {row['major_raw']!r:55s} → {row['major']!r}{flag}")
print(f"\nDistinct cleaned majors: {df['major'].nunique()}")
print(f"Double-major rows: {df['is_double'].sum()}")
df["major"].value_counts()

Row-by-row mapping (raw → cleaned, is_double):
  row  0: 'Mathematics'                                           → 'Mathematics'
  row  1: 'Economics modified with Computer Science, German Studies' → 'Economics'
  row  2: 'Economics '                                            → 'Economics'
  row  3: 'Gov and Econ double major'                             → 'Government' *double*
  row  4: 'Computer Science'                                      → 'Computer Science'
  row  5: 'Math mod Econ'                                         → 'Mathematics'
  row  6: 'Computer Science & Math'                               → 'Computer Science' *double*
  row  7: 'Economics '                                            → 'Economics'
  row  8: 'Math'                                                  → 'Mathematics'
  row  9: 'Economics '                                            → 'Economics'
  row 10: 'Economics'                                             → 'Economics'
  row 11: 'Engineering + Physic

major
Economics                14
Mathematics               5
Computer Science          4
Engineering               4
Government                3
QSS                       3
PPE                       2
Biology                   2
Earth Sciences            1
Environmental Studies     1
Religion                  1
Biochemistry              1
History                   1
Name: count, dtype: int64

## Ordinal encoding (w, q7, q9)

In [5]:
W_MAP = {
    "Not important at all": 1,
    "Slightly important": 2,
    "Moderately important": 3,
    "Very important": 4,
    "Extremely important": 5,
}
Q7_MAP = {
    "Never": 0,
    "Yes, once or twice": 1,
    "Yes, multiple times": 2,
}
Q9_MAP = {
    "Strongly no": -2,
    "Somewhat no": -1,
    "Neutral": 0,
    "Somewhat yes": 1,
    "Strongly yes": 2,
}

df["w"] = df["gpa_importance"].str.strip().map(W_MAP)
df["q7_score"] = df["q7_avoided"].str.strip().map(Q7_MAP)
df["q9_score"] = df["q9_env"].str.strip().map(Q9_MAP)

for col, src in [("w", "gpa_importance"), ("q7_score", "q7_avoided"), ("q9_score", "q9_env")]:
    unmapped = df.loc[df[col].isna(), src].dropna().unique()
    if len(unmapped):
        print(f"WARNING — unmapped values in {src}: {list(unmapped)}")
    else:
        print(f"OK — {col} encoded with no unmapped values")

OK — w encoded with no unmapped values
OK — q7_score encoded with no unmapped values
OK — q9_score encoded with no unmapped values


## Compute s and finalize

In [6]:
df["courses"] = pd.to_numeric(df["courses"], errors="coerce")
df["layups"] = pd.to_numeric(df["layups"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df["s"] = df["layups"] / df["courses"]
out_of_range = df[(df["s"] > 1) | (df["s"] < 0)]
if len(out_of_range):
    print("WARNING — rows with s outside [0,1]:")
    print(out_of_range[["layups", "courses", "s"]])
else:
    print(f"s ∈ [{df['s'].min():.3f}, {df['s'].max():.3f}] — all valid")

df["strategy_self_missing"] = (
    df["strategy_self"].isna() | (df["strategy_self"].astype(str).str.strip() == "")
)
df["q10_missing"] = (
    df["q10_counterfactual"].isna() | (df["q10_counterfactual"].astype(str).str.strip() == "")
)

s ∈ [0.000, 0.778] — all valid


In [7]:
print(f"Final shape: {df.shape}")
print("\nMajor counts:")
print(df["major"].value_counts())
print("\nTrack counts:")
print(df["track"].value_counts())
print("\nw distribution:")
print(df["w"].value_counts().sort_index())
print(f"\nMissing strategy_self: {df['strategy_self_missing'].sum()}")
print(f"Missing q10:           {df['q10_missing'].sum()}")

Final shape: (42, 19)

Major counts:
major
Economics                14
Mathematics               5
Computer Science          4
Engineering               4
Government                3
QSS                       3
PPE                       2
Biology                   2
Earth Sciences            1
Environmental Studies     1
Religion                  1
Biochemistry              1
History                   1
Name: count, dtype: int64

Track counts:
track
Finance/consulting       24
Tech/engineering          5
Undecided/other           5
Graduate/ PhD program     3
Government/ nonprofit     3
Pre-med                   2
Name: count, dtype: int64

w distribution:
w
1     1
2     6
3    10
4    18
5     7
Name: count, dtype: int64

Missing strategy_self: 1
Missing q10:           7


In [8]:
cols = [
    "timestamp", "year", "major", "major_raw", "is_double", "track",
    "courses", "layups", "s",
    "gpa_importance", "w",
    "q7_avoided", "q7_score",
    "q9_env", "q9_score",
    "strategy_self", "strategy_self_missing",
    "q10_counterfactual", "q10_missing",
]
df[cols].to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(df)} rows × {len(cols)} cols)")

Wrote ../data/cleaned_responses.csv (42 rows × 19 cols)
